### Chatbot with message History


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B50109F0B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B500193D40>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")])

AIMessage(content='Hello Kheman! 👋 Great to meet you. How’s engineering life treating you so far? Whether you’re tackling tough problem sets, working on a cool project, or just curious about a particular branch of engineering, I’m here to help. Let me know what you’d like to chat about!', additional_kwargs={'reasoning_content': 'The user says: "Hi My name is Kheman Saru and I am currently studying engineering". Likely they want a response, maybe a greeting, ask about engineering field, etc. As ChatGPT, respond politely, ask how can I help, maybe talk about engineering topics.\n\nNo disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 86, 'total_tokens': 225, 'completion_time': 0.29124718, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.003978316, 'prompt_tokens_details': None, 'queue_time': 0.045118603, 'total_time': 0.295225496}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [5]:
config = {"configurable":{"session_id":"chat_1"}}

In [6]:
response = with_message_history.invoke(
            [HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")],
            config=config
        )   

In [7]:
response.content

'Hello Kheman! 👋 It’s great to meet you. How’s engineering life treating you so far? What branch are you studying (mechanical, electrical, civil, computer, etc.)? If you have any questions, need study tips, project ideas, or just want to chat about engineering topics, I’m here to help!'

In [8]:
with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config
        )   

AIMessage(content='Your name is **Kheman\u202fSaru**.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" The user previously said "Hi My name is Kheman Saru and I am currently studying engineering". So answer: Kheman Saru.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 170, 'total_tokens': 231, 'completion_time': 0.127627867, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.006801709, 'prompt_tokens_details': None, 'queue_time': 0.04510337, 'total_time': 0.134429576}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cbece-8b6a-7823-8e82-4d0a8482c994-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 61, 'total_tokens': 231, 'output_token_details': {'reasoning': 40}})

In [9]:
config1 = {"configurable":{"session_id":"chat_2"}}
response = with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config1
        )   
response.content

'I’m sorry, but I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [10]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system""You are an ai assistant.Answer all the query like a teacher with nice tone"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt|model


In [12]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is Kheman")]})


AIMessage(content='Hello, Kheman! It’s wonderful to meet you. 😊  \n\nI’m here to help with any questions you have—whether it’s about a school subject, a hobby you’re exploring, or just something you’re curious about. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi my name is Kheman". The system says "You are an ai assistant. Answer all the query like a teacher with nice tone". So respond in a teacher-like nice tone, greeting them, maybe ask about them or how can I help. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 99, 'total_tokens': 223, 'completion_time': 0.260141089, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.003666278, 'prompt_tokens_details': None, 'queue_time': 0.045210111, 'total_time': 0.263807367}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_pr

In [13]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [17]:
config = {"configurable":{"session_id":"chat_4"}}
response=with_message_history.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)
response

AIMessage(content='I’m not sure what your name is just yet—could you let me know? Once I know it, I’ll be happy to address you properly!', additional_kwargs={'reasoning_content': 'The user says: "systemYou are an ai assistant.Answer all the query like a teacher with nice tone". Then they ask: "what is my name?" We have no prior context with the user\'s name. So we must respond politely, indicating we don\'t know the name and ask them to tell us. Must answer like a teacher with nice tone. So say something like: "I’m not sure what your name is yet, could you tell me?" Provide friendly tone.'}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 97, 'total_tokens': 234, 'completion_time': 0.306516545, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_time': 0.014963512, 'prompt_tokens_details': None, 'queue_time': 0.046865448, 'total_time': 0.321480057}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_d